# 5b Compare Proposals Rephrased

This notebook runs the style-controlled proposal analyses on the rephrased-text branch.

Within each condition it produces both:
- per-model Human vs model contrasts
- pooled Human vs All-AI contrasts

It reuses the prepared outputs from `4a_prepare_proposal_for_analysis.ipynb` and the human review-score overlays from `4b_prepare_review_for_analysis.ipynb`.

In [ ]:
CONDITIONS_TO_RUN = ['baseline', 'one_at_a_time', 'persona']
TEXT_VERSION = 'rephrased'

N_BOOT = 1000
BOOT_SEED = 42
N_SUBSAMPLE = 23
N_PERM = 10000
STYLE_PERM = 1000
GRID_BINS = 8


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from proposal_generation import find_project_root
from proposal_comparison import (
    PRIMARY_DIVERSITY_METRICS,
    build_proposal_metrics_master,
    compare_human_vs_group_metrics,
    compute_literature_self_knn_cache,
    cross_condition_summary_table,
    generate_or_load_bootstrap_samples,
    get_group_indices,
    load_condition_analysis_inputs,
    pooled_bootstrap_metric_distribution,
    run_simple_cluster_analysis,
    run_simple_topic_analysis,
    simple_style_features,
    style_classifier_permutation,
)

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
sns.set_theme(style='whitegrid', context='talk')
LITERATURE_EMBEDDINGS_PATH = PROJECT_ROOT / 'data' / 'embeddings' / 'literature' / 'relevant_literature_embeddings.pkl'
print(f'Project root: {PROJECT_ROOT}')
print(f'Conditions to run: {CONDITIONS_TO_RUN}')


In [ ]:
condition_results = {}
cross_condition_pooled_rows = []
cross_condition_per_model_rows = []

for condition in CONDITIONS_TO_RUN:
    print(f'\n=== 5b rephrased proposal comparison: {condition} ===')
    analysis = load_condition_analysis_inputs(PROJECT_ROOT, condition, text_version=TEXT_VERSION)
    group_idx = get_group_indices(analysis.proposal_master)
    human_idx = group_idx['Human']
    ai_idx = group_idx['All AI']

    if len(human_idx) != 23:
        raise RuntimeError(f'{condition}: expected 23 Human proposals, found {len(human_idx)}')
    if len(ai_idx) != 69:
        raise RuntimeError(f'{condition}: expected 69 pooled AI proposals, found {len(ai_idx)}')
    for model_label in ['Claude', 'Gemini', 'GPT']:
        if len(group_idx.get(model_label, [])) != 23:
            raise RuntimeError(f'{condition}: expected 23 proposals for {model_label}, found {len(group_idx.get(model_label, []))}')

    tables_root = PROJECT_ROOT / 'results' / 'tables' / condition / 'proposals' / TEXT_VERSION
    figures_root = PROJECT_ROOT / 'results' / 'figures' / condition / 'proposals' / TEXT_VERSION
    per_model_tables_dir = tables_root / 'per_model'
    per_model_figures_dir = figures_root / 'per_model'
    pooled_tables_dir = tables_root / 'all_ai'
    pooled_figures_dir = figures_root / 'all_ai'
    shared_cache_dir = tables_root / 'shared_cache'
    for p in [per_model_tables_dir, per_model_figures_dir, pooled_tables_dir, pooled_figures_dir, shared_cache_dir]:
        p.mkdir(parents=True, exist_ok=True)

    bootstrap_path_original = PROJECT_ROOT / 'results' / 'tables' / condition / 'proposals' / 'original' / 'shared_cache' / f'bootstrap_ai_idx_samples_n{N_SUBSAMPLE}_seed{BOOT_SEED}.npy'
    bootstrap_path_rephrased = shared_cache_dir / f'bootstrap_ai_idx_samples_n{N_SUBSAMPLE}_seed{BOOT_SEED}.npy'
    if bootstrap_path_original.exists() and analysis.proposal_master['proposal_uid'].astype(str).tolist() == pd.read_csv(PROJECT_ROOT / 'data' / 'prepared' / condition / 'proposals' / 'original' / 'proposal_master.csv')['proposal_uid'].astype(str).tolist():
        bootstrap_ai_idx_samples = np.load(bootstrap_path_original)
        np.save(bootstrap_path_rephrased, bootstrap_ai_idx_samples)
    else:
        bootstrap_ai_idx_samples = generate_or_load_bootstrap_samples(
            ai_idx,
            output_path=bootstrap_path_rephrased,
            n_boot=N_BOOT,
            n_subsample=N_SUBSAMPLE,
            seed=BOOT_SEED,
        )

    lit_self_knn_path = shared_cache_dir / 'lit_knn_distances_50.npy'
    lit_knn_distances_50 = compute_literature_self_knn_cache(
        LITERATURE_EMBEDDINGS_PATH,
        output_path=lit_self_knn_path,
        k=50,
    )
    lit_mean_knn_10 = lit_knn_distances_50[:, :10].mean(axis=1)

    proposal_metrics_master = build_proposal_metrics_master(analysis, lit_self_knn_mean10=lit_mean_knn_10)
    proposal_metrics_master.to_csv(pooled_tables_dir / 'proposal_metrics_master.csv', index=False)

    topic_df, topic_summary = run_simple_topic_analysis(analysis.proposal_master)
    topic_df.to_csv(shared_cache_dir / 'topic_assignments.csv', index=False)
    with open(shared_cache_dir / 'topic_distribution_summary.json', 'w') as f:
        json.dump(topic_summary, f, indent=2)

    cluster_df, cluster_summary = run_simple_cluster_analysis(
        np.asarray(analysis.full_embeddings['embeddings']),
        analysis.proposal_master,
    )
    cluster_df.to_csv(shared_cache_dir / 'cluster_assignments.csv', index=False)
    with open(shared_cache_dir / 'cluster_segregation_summary.json', 'w') as f:
        json.dump(cluster_summary, f, indent=2)

    style_X = simple_style_features(analysis.proposal_master['full_text'])
    style_y = (analysis.proposal_master['source_type'] == 'ai').astype(int).to_numpy()
    style_result = style_classifier_permutation(style_X, style_y, n_perm=STYLE_PERM, seed=BOOT_SEED)
    with open(shared_cache_dir / 'style_classifier_summary.json', 'w') as f:
        json.dump(style_result, f, indent=2)

    per_model_rows = []
    for model_label in ['Claude', 'Gemini', 'GPT']:
        model_idx = group_idx[model_label]
        metrics_df = compare_human_vs_group_metrics(
            analysis.pairwise_full,
            human_idx,
            model_idx,
            coords=analysis.umap2d,
            metric_names=PRIMARY_DIVERSITY_METRICS + ['mst_dispersion'],
            n_perm=N_PERM,
            seed=BOOT_SEED,
        )
        metrics_df['condition'] = condition
        metrics_df['comparison_group'] = model_label
        per_model_rows.append(metrics_df)
        for _, row in metrics_df.iterrows():
            cross_condition_per_model_rows.append(
                {
                    'condition': condition,
                    'comparison_group': model_label,
                    'metric': row['metric'],
                    'effect_human_minus_group': row['effect_human_minus_group'],
                    'permutation_p_value': row['permutation_p_value'],
                    'analysis_family': 'proposal_space_diversity',
                }
            )
    per_model_summary_df = pd.concat(per_model_rows, ignore_index=True)
    per_model_summary_df.to_csv(per_model_tables_dir / 'diversity_summary_human_vs_per_model.csv', index=False)

    pooled_rows = []
    bootstrap_frames = []
    for metric_name in PRIMARY_DIVERSITY_METRICS + ['mst_dispersion']:
        boot_df = pooled_bootstrap_metric_distribution(
            analysis.pairwise_full,
            human_idx,
            bootstrap_ai_idx_samples,
            metric_name=metric_name,
            coords=analysis.umap2d,
        )
        pooled_metric_df = compare_human_vs_group_metrics(
            analysis.pairwise_full,
            human_idx,
            ai_idx,
            coords=analysis.umap2d,
            metric_names=[metric_name],
            n_perm=N_PERM,
            seed=BOOT_SEED,
        )
        pooled_row = pooled_metric_df.iloc[0].to_dict()
        pooled_row.update(
            {
                'condition': condition,
                'human_value': float(boot_df['human_value'].iloc[0]),
                'ai_boot_mean': float(boot_df['ai_value'].mean()),
                'ai_boot_sd': float(boot_df['ai_value'].std()),
            }
        )
        pooled_rows.append(pooled_row)
        bootstrap_frames.append(boot_df.assign(condition=condition))
        cross_condition_pooled_rows.append(
            {
                'condition': condition,
                'metric': metric_name,
                'effect_human_minus_ai': pooled_row['effect_human_minus_group'],
                'permutation_p_value': pooled_row['permutation_p_value'],
                'analysis_family': 'proposal_space_diversity',
            }
        )
    pooled_summary_df = pd.DataFrame(pooled_rows)
    pooled_summary_df.to_csv(pooled_tables_dir / 'diversity_summary_human_vs_allai.csv', index=False)
    pd.concat(bootstrap_frames, ignore_index=True).to_csv(pooled_tables_dir / 'proposal_metrics_summary_human_vs_allai.csv', index=False)

    metric_corr_df = proposal_metrics_master[
        ['mean_pairwise_distance', 'nearest_neighbor_distance', 'literature_element_novelty_k1', 'literature_mean_knn_novelty_k10', 'literature_local_density_normalized_novelty']
    ].corr(numeric_only=True)
    metric_corr_df.to_csv(pooled_tables_dir / 'proposal_diversity_metric_correlations.csv')

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=per_model_summary_df[per_model_summary_df['metric'].isin(PRIMARY_DIVERSITY_METRICS)], x='comparison_group', y='effect_human_minus_group', hue='metric', ax=ax)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title(f'{condition}: per-model diversity effects (rephrased proposals)')
    ax.set_ylabel('Effect (Human minus model)')
    ax.set_xlabel('Model')
    plt.tight_layout()
    fig.savefig(per_model_figures_dir / 'diversity_effects_human_vs_per_model.png', dpi=200)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=pooled_summary_df[pooled_summary_df['metric'].isin(PRIMARY_DIVERSITY_METRICS)], x='metric', y='effect_human_minus_group', ax=ax, color='#4A90E2')
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title(f'{condition}: pooled diversity effects (rephrased proposals)')
    ax.set_ylabel('Effect (Human minus All AI)')
    ax.set_xlabel('Metric')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    fig.savefig(pooled_figures_dir / 'diversity_effects_human_vs_allai.png', dpi=200)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 6))
    plot_df = analysis.proposal_master.copy()
    plot_df['umap_x'] = analysis.umap2d[:, 0]
    plot_df['umap_y'] = analysis.umap2d[:, 1]
    sns.scatterplot(data=plot_df, x='umap_x', y='umap_y', hue='source_group', style='source_type', ax=ax)
    ax.set_title(f'{condition}: rephrased proposal-space UMAP')
    plt.tight_layout()
    fig.savefig(pooled_figures_dir / 'proposal_space_umap_rephrased.png', dpi=200)
    plt.close(fig)

    condition_results[condition] = {
        'analysis': analysis,
        'proposal_metrics_master': proposal_metrics_master,
        'per_model_summary_df': per_model_summary_df,
        'pooled_summary_df': pooled_summary_df,
        'topic_summary': topic_summary,
        'cluster_summary': cluster_summary,
        'style_result': style_result,
        'per_model_tables_dir': per_model_tables_dir,
        'pooled_tables_dir': pooled_tables_dir,
    }
    print(f'Finished rephrased proposal analysis for {condition}')


In [ ]:
cross_tables_dir = PROJECT_ROOT / 'results' / 'tables' / 'proposals' / TEXT_VERSION / 'cross_condition'
cross_figures_dir = PROJECT_ROOT / 'results' / 'figures' / 'proposals' / TEXT_VERSION / 'cross_condition'
cross_tables_dir.mkdir(parents=True, exist_ok=True)
cross_figures_dir.mkdir(parents=True, exist_ok=True)

cross_pooled_df = cross_condition_summary_table(cross_condition_pooled_rows)
cross_per_model_df = cross_condition_summary_table(cross_condition_per_model_rows)
cross_pooled_df.to_csv(cross_tables_dir / 'pooled_human_vs_allai_cross_condition_summary.csv', index=False)
cross_per_model_df.to_csv(cross_tables_dir / 'per_model_human_vs_model_cross_condition_summary.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=cross_pooled_df[cross_pooled_df['metric'].isin(PRIMARY_DIVERSITY_METRICS)], x='condition', y='effect_human_minus_ai', hue='metric', ax=ax)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Cross-condition pooled primary diversity effects (rephrased proposals)')
ax.set_ylabel('Effect (Human minus All AI)')
ax.set_xlabel('Condition')
plt.tight_layout()
fig.savefig(cross_figures_dir / 'pooled_primary_diversity_effects_cross_condition.png', dpi=200)
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=cross_per_model_df[cross_per_model_df['metric'].isin(PRIMARY_DIVERSITY_METRICS)], x='condition', y='effect_human_minus_group', hue='comparison_group', ax=ax)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Cross-condition per-model primary diversity effects (rephrased proposals)')
ax.set_ylabel('Effect (Human minus model)')
ax.set_xlabel('Condition')
plt.tight_layout()
fig.savefig(cross_figures_dir / 'per_model_primary_diversity_effects_cross_condition.png', dpi=200)
plt.close(fig)

cross_pooled_df
